<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.1-token-economics/notebooks/GCP_Capstone_2.1_Token_Economics.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.1 Token Economics — The Currency of AI
**Netsetos GenAI Engineering — GCP Capstone**

Master tokenization, multilingual costs, context budgeting, and build the token economics module.


## Setup


In [ ]:
!pip install -q "google-genai[local-tokenizer]==2.21.0"
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
LOCATION = 'global'  # Gemini 3.x generation (count_tokens/generate_content) is served from the global endpoint
INR = 85

from google import genai
from google.genai import types
client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)


## Cell 1: LocalTokenizer — Offline Token Counting


In [ ]:
from google.genai.local_tokenizer import LocalTokenizer

try:
    tok = LocalTokenizer(model_name='gemini-3-flash-preview')  # LocalTokenizer's registry only lists the base Gemini-3 name; the tokenizer is shared across the Gemini 3 family
except Exception as e:
    print('LocalTokenizer registry has no local model for this ID:', e)
    print('Falling back to client.models.count_tokens (an API call, still free)')
    tok = None

def count(text):
    if tok is not None:
        return tok.count_tokens(text).total_tokens
    return client.models.count_tokens(
        model='gemini-3.6-flash', contents=text).total_tokens

texts = [
    'Hello world!',
    'The transformer architecture uses self-attention.',
    'https://cloud.google.com/vertex-ai/docs',
    'def hello(): print("Hi")',
    '{"name": "Sart", "city": "Hyderabad"}',
]

print(f"  {'Text':<45} {'Chars':>5} {'Tokens':>6} {'Ratio':>6}")
for t in texts:
    n = count(t)
    print(f"  {t[:42]:<45} {len(t):>5} {n:>6} {len(t)/n:>4.1f}:1")


## Cell 2: Hindi/Telugu Token Tax


In [ ]:
sentences = {
    'English': 'Artificial intelligence is changing the world.',
    'Hindi': '\u0915\u0943\u0924\u094d\u0930\u093f\u092e \u092c\u0941\u0926\u094d\u0927\u093f\u092e\u0924\u094d\u0924\u093e \u0926\u0941\u0928\u093f\u092f\u093e \u0915\u094b \u092c\u0926\u0932 \u0930\u0939\u0940 \u0939\u0948\u0964',
    'Telugu': '\u0c06\u0c30\u0c4d\u0c1f\u0c3f\u0c2b\u0c3f\u0c37\u0c3f\u0c2f\u0c32\u0c4d \u0c07\u0c02\u0c1f\u0c46\u0c32\u0c3f\u0c1c\u0c46\u0c28\u0c4d\u0c38\u0c4d \u0c2a\u0c4d\u0c30\u0c2a\u0c02\u0c1a\u0c02\u0c32\u0c4b \u0c2e\u0c3e\u0c30\u0c4d\u0c2a\u0c41\u0c32\u0c41 \u0c1a\u0c46\u0c02\u0c26\u0c41\u0c24\u0c4b\u0c02\u0c26\u0c3f.',
}

en_tok = None
for lang, text in sentences.items():
    r = client.models.count_tokens(model='gemini-3.6-flash', contents=text)
    if lang == 'English': en_tok = r.total_tokens
    mult = r.total_tokens / en_tok
    cost_10k = 10000 * r.total_tokens * 1.50 / 1e6 * INR
    print(f'  {lang:<10} {r.total_tokens:>4} tokens | {mult:.1f}x | Rs {cost_10k:.2f}/10K queries')


## Cell 3: Multimodal Token Costs


In [ ]:
# MEASURE the image token cost - do not assume a constant.
# Vertex AI has no Files API (client.files.upload is Gemini Developer API
# only), so build the Part from local bytes or from a GCS URI.
import os
from google.genai import types

IMAGE_PATH = 'sample_chart.png'  # upload your own image to Colab first

if os.path.exists(IMAGE_PATH):
    image_part = types.Part.from_bytes(
        data=open(IMAGE_PATH, 'rb').read(),
        mime_type='image/png',
    )
    r = client.models.count_tokens(          # global client
        model='gemini-3.6-flash',
        contents=[image_part, 'Describe this chart'],
    )
    print(f'Measured image + prompt tokens: {r.total_tokens}')
    print(f'  at gemini-3.6-flash standard input rate $1.50/1M (from 2027-01-01), USD_INR=85: Rs {r.total_tokens * 1.50 / 1e6 * INR:.4f}')
else:
    print(f'{IMAGE_PATH} not found - upload an image to Colab and re-run this cell.')

# From a GCS object instead of local bytes:
# image_part = types.Part.from_uri(
#     file_uri='gs://documind-ai-YOUR-ID-media/sample_chart.png',
#     mime_type='image/png')

# Image tiers below were measured with count_tokens on gemini-3.6-flash -
# re-run the cell for your own file, tiers change between model versions.
for name, tokens in [('LOW',280),('MEDIUM',560),('HIGH',1120),('ULTRA_HIGH',2240)]:
    cost = tokens * 1.50 / 1e6 * INR
    print(f'  {name:<15} {tokens:>5} tokens | Rs {cost:.4f}/image')

# Audio
for mins in [1, 5, 60]:
    tokens = mins * 60 * 32
    cost = tokens * 1.50 / 1e6 * INR  # audio rate ($1.50/M on 3.x flash)
    print(f'  {mins:>2} min audio   {tokens:>7,} tokens | Rs {cost:.4f}')

# PDF - per-page figure measured with count_tokens on gemini-3.6-flash - re-run the cell for your own file
for pages in [1, 10, 100]:
    tokens = pages * 258
    cost = tokens * 1.50 / 1e6 * INR
    print(f'  {pages:>3} PDF pages  {tokens:>7,} tokens | Rs {cost:.4f}')

## Cell 4: Context Budget Calculator

Lesson 4.5 turns this calculator into a TokenBudget the RAG pipeline enforces on every request.


In [ ]:
def budget_context(client, model, system='', history=None, rag_context='', query='', margin=200):
    info = client.models.get(model=model)
    max_in = info.input_token_limit or 1_048_576
    max_out = info.output_token_limit or 65_536
    parts = {}
    for name, txt in [('system',system),('rag',rag_context),('query',query)]:
        parts[name] = client.models.count_tokens(model=model, contents=txt).total_tokens if txt else 0
    parts['history'] = client.models.count_tokens(model=model, contents=history).total_tokens if history else 0
    total = sum(parts.values()) + margin
    avail = min(max_out, max_in - total)
    print(f'Context Budget for {model}:')
    for k, v in parts.items(): print(f'  {k:<12} {v:>8,} tokens')
    print(f'  {"margin":<12} {margin:>8,} tokens')
    print(f'  {"TOTAL":<12} {total:>8,} tokens ({total/max_in*100:.2f}%)')
    print(f'  {"Available":<12} {avail:>8,} for output')
    return {'total': total, 'available': avail}

budget_context(client, 'gemini-3.6-flash',
    system='You are DocuMind AI. Cite sources.',
    rag_context='[Document chunks]' * 100,
    query='Summarize key findings.')


## Cell 5: 4-Scenario Cost Comparison


In [ ]:
PRICING = {
    '3.1 Flash-Lite': (0.25, 1.50),
    '3.6 Flash': (1.50, 7.50),
    '3.1 Pro': (2.00, 12.00),
}

scenarios = [
    ('Simple Q&A', 20, 50),
    ('RAG Query', 2000, 500),
    ('Doc Analysis', 50000, 1000),
    ('10-turn Chat', 17400, 2000),
]

for name, inp, out in scenarios:
    print(f'\n{name} ({inp:,} in + {out:,} out):')
    for model, (ip, op) in PRICING.items():
        cost = (inp*ip + out*op)/1e6*INR
        print(f'  {model:<16} Rs {cost:.4f}')


## Cell 6: Multi-Turn Cost Growth


In [ ]:
sys_tok = 200; tok_per_turn = 400; out_per_turn = 200

print(f"  {'Turn':>4} {'Full Rs':>10} {'Window Rs':>10} {'Savings':>9}")
cum_full = cum_win = 0
for turn in range(1, 21):
    full_in = sys_tok + (turn-1)*tok_per_turn + 200
    full_cost = (full_in*1.50 + out_per_turn*7.50)/1e6*INR
    cum_full += full_cost
    win_in = sys_tok + min(turn-1,5)*tok_per_turn + 200
    win_cost = (win_in*1.50 + out_per_turn*7.50)/1e6*INR
    cum_win += win_cost
    sav = (cum_full-cum_win)/cum_full*100 if cum_full else 0
    print(f"  {turn:>4} Rs{cum_full:>9.4f} Rs{cum_win:>9.4f}  {sav:>6.1f}%")
print(f"\n  Windowing saves: Rs{cum_full-cum_win:.4f} ({(cum_full-cum_win)/cum_full*100:.0f}%)")


## Cell 7: The token economics module (the kit prices in services/rag-api/cost.py)


In [ ]:
# DocuMind token economics - the lesson's module; the lane prices answers in deploy/services/rag-api/cost.py
from google.genai.local_tokenizer import LocalTokenizer

PRICING = {
    'gemini-3.1-flash-lite': {'input':0.25,'output':1.50},
    'gemini-3.6-flash': {'input':1.50,'output':7.50},
    'gemini-3.1-pro-preview': {'input':2.00,'output':12.00},
}
_tok = None
try:
    _tok = LocalTokenizer(model_name='gemini-3-flash-preview')
except Exception as e:
    print('LocalTokenizer registry has no local model for this ID:', e)
    print('Falling back to client.models.count_tokens (an API call, still free)')

def count_local(text, client=None):
    """Offline when the local tokenizer loaded; API count otherwise."""
    if _tok is not None:
        return _tok.count_tokens(text).total_tokens
    if client is None:
        raise RuntimeError(
            'LocalTokenizer unavailable - pass client=... so count_local can use the API counter')
    return client.models.count_tokens(
        model='gemini-3.6-flash', contents=text).total_tokens

def cost_inr(meta, model='gemini-3.6-flash'):
    p = PRICING.get(model, PRICING['gemini-3.6-flash'])
    think = (getattr(meta, 'thoughts_token_count', 0) or 0)
    i = meta.prompt_token_count * p['input'] / 1e6
    o = (meta.candidates_token_count + think) * p['output'] / 1e6
    return {'inr': round((i+o)*85, 4), 'usd': round(i+o, 6)}

def monthly_projection(qpd, avg_in, avg_out, mix={'gemini-3.1-flash-lite':0.6,'gemini-3.6-flash':0.3,'gemini-3.1-pro-preview':0.1}):
    monthly = qpd * 30; total = 0
    for model, pct in mix.items():
        p = PRICING[model]; q = monthly * pct
        total += (q*avg_in*p['input'] + q*avg_out*p['output'])/1e6
    return {'usd':round(total,2), 'inr':round(total*85,2), 'months':round(500/total,1) if total>0 else float('inf')}

print('Local count:', count_local('Hello Hyderabad!', client))
r = monthly_projection(100, 2000, 500)
print(f"Monthly: ${r['usd']}/mo (Rs {r['inr']}) | $500 lasts {r['months']} months")


## ✅ Lesson 2.1 Complete!

- ✅ SentencePiece Unigram tokenizer understood
- ✅ Hindi/Telugu 2x token tax discovered
- ✅ Multimodal token costs mapped
- ✅ Context budget calculator built
- ✅ 3-model pricing compared in INR
- ✅ Multi-turn cost growth simulated
- ✅ The token economics module, the lesson's version of deploy/services/rag-api/cost.py

**Next: Lesson 2.2 — Embeddings: Text to Vectors**
